# glass1: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('18_glass1_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,65.266,57.374,74.225,47.500,90.556,69.028,71.161,79.555,61.320
smote,68.530,61.316,59.501,66.417,72.870,69.644,67.907,79.170,64.923
bsmote,68.282,60.468,57.774,65.708,73.161,69.435,67.146,79.825,64.375
ros,70.289,62.750,61.824,65.833,76.799,71.316,70.223,80.470,66.520
adasyn,69.312,61.764,57.951,67.708,72.460,70.084,66.366,79.143,65.538
casb,70.555,62.973,62.002,65.750,77.169,71.460,67.866,79.294,66.764
casb_relaxed,68.614,61.135,59.081,65.042,73.836,69.439,69.325,80.204,64.874
casb_edited,71.131,65.249,58.565,76.333,68.069,72.201,68.294,81.170,68.190
casb_ros,69.210,61.979,59.268,67.042,73.148,70.095,67.534,79.132,65.594


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,73.665,67.791,77.161,60.750,89.788,75.269,78.530,83.527,70.728
smote,75.722,69.796,72.403,68.583,84.365,76.474,78.980,84.080,72.759
bsmote,75.590,69.666,73.269,67.208,85.437,76.322,78.120,84.011,72.628
ros,75.539,69.587,70.930,69.875,82.500,76.188,77.610,83.511,72.563
adasyn,76.060,69.883,70.783,70.583,82.910,76.747,75.170,83.569,72.971
casb,75.846,69.885,73.233,67.958,85.463,76.711,77.300,83.639,72.866
casb_relaxed,73.899,67.480,72.182,64.000,85.833,74.917,79.338,84.308,70.690
casb_edited,76.467,70.556,68.829,73.792,79.974,76.883,77.003,84.267,73.511
casb_ros,75.806,69.777,73.597,67.333,86.177,76.755,78.157,83.925,72.791


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,77.015,71.889,80.149,66.000,90.556,78.278,84.231,88.733,74.452
smote,77.321,72.146,78.348,68.000,88.730,78.365,84.213,89.227,74.734
bsmote,80.178,75.774,80.413,73.000,88.730,80.865,85.247,89.135,77.976
ros,77.186,71.802,77.397,67.958,88.360,78.159,84.325,89.105,74.494
adasyn,79.138,73.803,75.526,73.167,86.177,79.672,83.695,89.315,76.470
casb,79.225,74.485,78.883,71.792,87.989,79.891,82.837,88.859,76.855
casb_relaxed,79.629,74.874,78.566,72.458,87.963,80.211,83.967,89.420,77.251
casb_edited,77.513,72.270,72.994,74.333,82.169,78.251,82.834,88.687,74.891
casb_ros,77.313,71.969,77.284,68.583,88.003,78.293,83.316,88.444,74.641


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,38.564,24.744,48.000,17.083,89.101,53.092,48.416,64.514,31.654
smote,61.616,55.593,45.867,71.042,53.968,62.505,48.116,63.438,58.604
bsmote,55.348,47.365,41.466,56.792,55.450,56.121,49.854,62.422,51.356
ros,63.261,57.138,47.810,72.583,56.534,64.559,49.416,64.929,60.200
adasyn,61.171,54.724,45.389,69.250,54.352,61.801,49.660,64.715,57.947
casb,58.929,51.241,44.809,61.375,57.976,59.676,49.321,64.554,55.085
casb_relaxed,60.874,54.048,45.815,67.083,56.468,61.776,48.910,64.886,57.461
casb_edited,60.910,53.526,45.875,65.167,58.294,61.730,50.664,65.884,57.218
casb_ros,58.487,50.714,44.395,60.750,57.976,59.363,50.037,64.961,54.600


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,63.627,55.449,69.884,47.833,90.000,68.917,70.584,79.082,59.538
smote,70.797,64.713,64.030,68.510,74.983,71.747,69.804,78.979,67.755
bsmote,69.849,63.318,63.230,65.677,75.694,70.686,70.092,78.848,66.584
ros,71.569,65.319,64.490,69.062,76.048,72.555,70.393,79.504,68.444
adasyn,71.420,65.044,62.412,70.177,73.975,72.076,68.723,79.185,68.232
casb,71.139,64.646,64.732,66.719,77.149,71.934,69.331,79.086,67.892
casb_relaxed,70.754,64.384,63.911,67.146,76.025,71.585,70.385,79.704,67.569
casb_edited,71.505,65.400,61.566,72.406,72.126,72.266,69.699,80.002,68.453
casb_ros,70.204,63.610,63.636,65.927,76.326,71.127,69.761,79.116,66.907


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,71.131,ros,70.289,0.842,WIN
1,f1,casb_edited,65.249,ros,62.750,2.499,WIN
2,precision,casb,62.002,ros,61.824,0.178,WIN
3,recall,casb_edited,76.333,adasyn,67.708,8.625,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,76.467,adasyn,76.060,0.407,WIN
1,f1,casb_edited,70.556,adasyn,69.883,0.673,WIN
2,precision,casb_ros,73.597,bsmote,73.269,0.327,WIN
3,recall,casb_edited,73.792,adasyn,70.583,3.208,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,79.629,bsmote,80.178,-0.548,LOSS
1,f1,casb_relaxed,74.874,bsmote,75.774,-0.900,LOSS
2,precision,casb,78.883,bsmote,80.413,-1.529,LOSS
3,recall,casb_edited,74.333,adasyn,73.167,1.167,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,71.505,ros,71.569,-0.064,LOSS
1,f1,casb_edited,65.400,ros,65.319,0.081,WIN
2,precision,casb,64.732,ros,64.490,0.242,WIN
3,recall,casb_edited,72.406,adasyn,70.177,2.229,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.577350,0.480000,0.666667,0.375000,0.888889,0.631944,0.723870,0.754630,0.528675,None
1,42,1,svm,smote,1.00,0.707107,0.648649,0.571429,0.750000,0.666667,0.708333,0.695867,0.752315,0.677878,None
2,42,1,svm,bsmote,1.00,0.695555,0.628571,0.578947,0.687500,0.703704,0.695602,0.635214,0.743056,0.662063,None
3,42,1,svm,ros,0.75,0.677003,0.600000,0.642857,0.562500,0.814815,0.688657,0.744677,0.759259,0.638502,None
4,42,1,svm,adasyn,1.00,0.707107,0.648649,0.571429,0.750000,0.666667,0.708333,0.659004,0.750000,0.677878,None
5,42,1,svm,casb,0.75,0.731247,0.666667,0.647059,0.687500,0.777778,0.732639,0.621063,0.752315,0.698957,None
6,42,1,svm,casb_relaxed,1.00,0.623610,0.585366,0.480000,0.750000,0.518519,0.634259,0.686437,0.770833,0.604488,None
7,42,1,svm,casb_edited,1.00,0.577350,0.558140,0.444444,0.750000,0.444444,0.597222,0.670271,0.766204,0.567745,None
8,42,1,svm,casb_ros,1.00,0.600925,0.571429,0.461538,0.750000,0.481481,0.615741,0.637188,0.738426,0.586177,None
9,42,1,knn,original,0.50,0.583333,0.482759,0.538462,0.437500,0.777778,0.607639,0.605494,0.685185,0.533046,None
